# state_conditioned_participation_drawdown_objective_v1

Notebook-first screen for the next training objective after `deep_cap_failure_attribution_v1`.

The prior diagnostic found a joint failure mode: rolling bad windows remain `underparticipation`, while strict OOS / walk-forward gate gaps are dominated by MaxDD. This notebook tests whether an explicit path-equity drawdown controller can be combined with a state-conditioned participation router before opening another PPO run.

This is a diagnostic target-path screen, not runtime promotion.

## Experiment Contract

- **Canonical baseline**: governed `v19reg_v22_budget_usage_bull_guard_v1` / current runtime composer family.
- **Direct comparison baseline**: `state_conditioned_objective_screen_v1` and `deep_cap_failure_attribution_v1`.
- **Role**: `diagnose`.
- **Changed layer**: `model_selection`, `reward`, and default-off reusable router scoring.
- **Target windows**: strict OOS plus `wf_2020`, `wf_2022`, `wf_2023_2024`.
- **Acceptance**: every scored window must clear `relative_return >= +30pp` and `MaxDD <= 15%`, and shared poor-window clusters must avoid underparticipation.
- **Expected trade-off**: lower participation may reduce MaxDD but must not collapse recovery participation.
- **Failure mode**: drawdown controller lowers MaxDD while reopening underparticipation; that is not PPO-ready.

## External Basis

- Qlib keeps this as a reproducible research-to-production artifact with persisted paths and replayable metrics.
- PBO/CSCV motivates keeping this as a walk-forward diagnostic rather than selecting from repeated screens as promotion.
- FinRL separates environment, reward, and action design; this notebook treats the controller as a reward/model-selection target, not a broker execution shortcut.
- Risk-sensitive / drawdown-aware RL literature motivates penalizing path drawdown directly when return-only or participation-only objectives overfit.

In [1]:
from __future__ import annotations

import json
import sys
from dataclasses import asdict, replace
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "AGENTS.md").exists() and (path / "src/soxl_quant").exists():
            return path
    raise RuntimeError("could not locate repository root")

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import (
    StateConditionedObjectiveConfig,
    daily_path_metrics,
    render_state_conditioned_objective_review,
    score_state_conditioned_objective,
    slice_daily_path_metrics,
    summarize_state_conditioned_objective,
)
from src.soxl_quant.app.state_conditioned_router import (
    StateConditionedRouterConfig,
    build_state_conditioned_router_frame,
    state_conditioned_router_metrics,
)

CARD = ROOT / "runtime/model_selection/card/state_conditioned_participation_drawdown_objective_v1"
CARD.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = CARD / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_EXCESS = 30.0
TARGET_MAX_DD = 15.0

WINDOWS = {
    "strict_oos": {
        "base": ROOT / "runtime/execution/card/current_coverage_residual_guard_refresh_v1/selected_strict_oos_daily.csv",
        "upside": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/recovery_reentry_conditional_vol_target_v1_soxl_daily.csv",
    },
    "wf_2020": {
        "base": ROOT / "runtime/execution/card/current_coverage_residual_guard_refresh_v1/selected_wf_2020_daily.csv",
        "upside": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2020/recovery_reentry_conditional_vol_target_v1_wf_2020_soxl_daily.csv",
    },
    "wf_2022": {
        "base": ROOT / "runtime/execution/card/current_coverage_residual_guard_refresh_v1/selected_wf_2022_daily.csv",
        "upside": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2022/recovery_reentry_conditional_vol_target_v1_wf_2022_soxl_daily.csv",
    },
    "wf_2023_2024": {
        "base": ROOT / "runtime/execution/card/current_coverage_residual_guard_refresh_v1/selected_wf_2023_2024_daily.csv",
        "upside": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2023_2024/recovery_reentry_conditional_vol_target_v1_wf_2023_2024_soxl_daily.csv",
    },
}
CLUSTERS = pd.read_csv(ROOT / "runtime/model_selection/card/deep_cap_failure_attribution_v1/aggregate/poor_window_clusters.csv")

EXTERNAL_REFERENCES = [
    {"label": "Qlib", "url": "https://github.com/microsoft/qlib", "note": "reproducible AI quant workflow"},
    {"label": "Qlib arXiv", "url": "https://arxiv.org/abs/2009.11189", "note": "data-learning-backtest-deployment workflow"},
    {"label": "Probability of Backtest Overfitting", "url": "https://doi.org/10.21314/JCF.2016.322", "note": "walk-forward discipline under repeated trials"},
    {"label": "FinRL", "url": "https://github.com/AI4Finance-Foundation/FinRL", "note": "DRL trading environment/reward/action separation"},
    {"label": "FinRL arXiv", "url": "https://arxiv.org/abs/2011.09607", "note": "risk-aware DRL trading framework reference"},
]

print(CARD.relative_to(ROOT))

runtime/model_selection/card/state_conditioned_participation_drawdown_objective_v1


## 1. Candidate Configs

Start from the lowest-loss router config selected by `state_conditioned_rebound_risk_router_v1`, then add a small number of explicit equity-drawdown controller variants. This is not a broad threshold grid; it tests whether the missing term identified by the prior attribution changes the objective score.

In [2]:
previous_router = json.loads((ROOT / "runtime/reward/card/state_conditioned_rebound_risk_router_v1/selection.json").read_text())
base_config = StateConditionedRouterConfig(**previous_router["selected_config"])

CONFIGS = {
    "selected_router_no_equity_dd": base_config,
    "selected_router_dd12_22_cap25": replace(base_config, equity_dd_start=0.12, equity_dd_full=0.22, equity_dd_min_cap=0.25),
    "selected_router_dd10_20_cap20": replace(base_config, equity_dd_start=0.10, equity_dd_full=0.20, equity_dd_min_cap=0.20),
    "selected_router_dd08_18_cap15": replace(base_config, equity_dd_start=0.08, equity_dd_full=0.18, equity_dd_min_cap=0.15),
}

config_rows = []
for label, config in CONFIGS.items():
    config_rows.append({"candidate": label, **asdict(config)})
config_df = pd.DataFrame(config_rows)
config_df.to_csv(CARD / "candidate_configs.csv", index=False)
display(config_df)

,candidate,upside_cap,rebound_mom21,rebound_volq,risk_cap,risk_dd63,risk_mom21,use_semis_stress,require_semis_widen,rebound_mom5,rebound_min_dd63,semis_ratio_z,semis_ratio_widen,semis_soxx_mom21,equity_dd_start,equity_dd_full,equity_dd_min_cap
0,selected_router_no_equity_dd,0.45,0.4,0.4,0.15,-0.2,0.0,True,False,0.02,-0.35,-0.75,-0.05,0.0,0.00,0.00,1.00
1,selected_router_dd12_22_cap25,0.45,0.4,0.4,0.15,-0.2,0.0,True,False,0.02,-0.35,-0.75,-0.05,0.0,0.12,0.22,0.25
2,selected_router_dd10_20_cap20,0.45,0.4,0.4,0.15,-0.2,0.0,True,False,0.02,-0.35,-0.75,-0.05,0.0,0.10,0.20,0.20
3,selected_router_dd08_18_cap15,0.45,0.4,0.4,0.15,-0.2,0.0,True,False,0.02,-0.35,-0.75,-0.05,0.0,0.08,0.18,0.15


## 2. Build Target Paths And Window Metrics

For each candidate, rebuild daily paths from the same base/upside windows, recompute NAV from routed positions, and then score strict OOS + walk-forward windows. Exported daily paths are diagnostic target paths only.

In [3]:
window_metrics_by_candidate: dict[str, dict[str, dict[str, object]]] = {}
cluster_metrics_by_candidate: dict[str, dict[str, dict[str, object]]] = {}
window_rows: list[dict[str, object]] = []
cluster_rows: list[dict[str, object]] = []
manifest_rows: list[dict[str, object]] = []

for candidate, config in CONFIGS.items():
    window_metrics_by_candidate[candidate] = {}
    cluster_metrics_by_candidate[candidate] = {}
    for window, paths in WINDOWS.items():
        base_daily = pd.read_csv(paths["base"])
        upside_daily = pd.read_csv(paths["upside"])
        frame = build_state_conditioned_router_frame(base_daily, upside_daily, config)
        frame["position"] = frame["router_position"]
        frame["candidate"] = candidate
        frame["window"] = window
        out_path = EXPORT_DIR / f"{candidate}_{window}_daily.csv"
        frame.to_csv(out_path, index=False)
        manifest_rows.append({
            "candidate": candidate,
            "window": window,
            "path": str(out_path.relative_to(ROOT)),
            "rows": int(len(frame)),
        })

        metrics = daily_path_metrics(frame)
        window_metrics_by_candidate[candidate][window] = metrics
        window_rows.append({"candidate": candidate, "window": window, **metrics})

        if window == "strict_oos":
            for cluster in CLUSTERS.to_dict("records"):
                cluster_id = str(cluster["cluster_id"])
                cmetrics = slice_daily_path_metrics(
                    frame,
                    start=str(cluster["cluster_start"]),
                    end=str(cluster["cluster_end"]),
                )
                cluster_metrics_by_candidate[candidate][cluster_id] = cmetrics
                cluster_rows.append({"candidate": candidate, "cluster_id": cluster_id, **cmetrics})

window_df = pd.DataFrame(window_rows)
cluster_df = pd.DataFrame(cluster_rows)
manifest_df = pd.DataFrame(manifest_rows)
window_df.to_csv(CARD / "window_metrics.csv", index=False)
cluster_df.to_csv(CARD / "cluster_metrics.csv", index=False)
manifest_df.to_csv(CARD / "candidate_path_manifest.csv", index=False)

display(window_df)
display(cluster_df)

,candidate,window,rows,start,end,return,buy_and_hold_return,relative_return,max_dd,avg_pos
0,selected_router_no_equity_dd,strict_oos,1605,2020-01-02,2026-05-21,484.586129,816.512500,-331.926371,31.932760,0.296688
1,selected_router_no_equity_dd,wf_2020,252,2020-01-02,2020-12-30,60.055301,58.093571,1.961730,24.815273,0.352449
2,selected_router_no_equity_dd,wf_2022,250,2022-01-03,2022-12-29,-15.515753,-86.588072,71.072319,16.700375,0.118209
3,selected_router_no_equity_dd,wf_2023_2024,251,2024-01-02,2024-12-30,42.805815,-0.356635,43.162450,15.975949,0.227884
4,selected_router_dd12_22_cap25,strict_oos,1605,2020-01-02,2026-05-21,459.300977,816.512500,-357.211523,31.932760,0.292015
5,selected_router_dd12_22_cap25,wf_2020,252,2020-01-02,2020-12-30,59.739252,58.093571,1.645681,24.373624,0.351277
6,selected_router_dd12_22_cap25,wf_2022,250,2022-01-03,2022-12-29,-15.515753,-86.588072,71.072319,16.700375,0.118209
7,selected_router_dd12_22_cap25,wf_2023_2024,251,2024-01-02,2024-12-30,42.805815,-0.356635,43.162450,15.975949,0.227884
8,selected_router_dd10_20_cap20,strict_oos,1605,2020-01-02,2026-05-21,293.918197,816.512500,-522.594303,31.978926,0.279867
9,selected_router_dd10_20_cap20,wf_2020,252,2020-01-02,2020-12-30,31.647548,58.093571,-26.446023,22.226483,0.318368


,candidate,cluster_id,rows,start,end,return,buy_and_hold_return,relative_return,max_dd,avg_pos
0,selected_router_no_equity_dd,C002,74,2025-04-04,2025-07-22,42.362882,201.489130,-159.126248,11.084311,0.449880
1,selected_router_no_equity_dd,C003,75,2026-02-04,2026-05-21,31.759225,235.382598,-203.623373,7.259839,0.243563
2,selected_router_no_equity_dd,C001,66,2020-03-18,2020-06-19,60.954643,178.461557,-117.506914,17.169856,0.555591
3,selected_router_dd12_22_cap25,C002,74,2025-04-04,2025-07-22,41.801664,201.489130,-159.687466,6.884687,0.413394
4,selected_router_dd12_22_cap25,C003,75,2026-02-04,2026-05-21,30.749508,235.382598,-204.633090,7.259839,0.240606
5,selected_router_dd12_22_cap25,C001,66,2020-03-18,2020-06-19,60.788220,178.461557,-117.673338,17.169856,0.555270
6,selected_router_dd10_20_cap20,C002,74,2025-04-04,2025-07-22,32.911693,201.489130,-168.577437,4.988088,0.351121
7,selected_router_dd10_20_cap20,C003,75,2026-02-04,2026-05-21,22.683544,235.382598,-212.699054,7.259839,0.183194
8,selected_router_dd10_20_cap20,C001,66,2020-03-18,2020-06-19,49.362740,178.461557,-129.098817,17.169856,0.540375
9,selected_router_dd08_18_cap15,C002,74,2025-04-04,2025-07-22,21.529385,201.489130,-179.959746,4.988088,0.221780


## 3. Score Joint Objective

The objective requires both window-level gate pass and cluster participation recovery. A lower loss without full gate pass is only a training-target hint.

In [4]:
objective_config = StateConditionedObjectiveConfig(
    target_relative_return=TARGET_EXCESS,
    target_max_dd=TARGET_MAX_DD,
    cluster_relative_floor=-30.0,
    cluster_participation_floor=0.35,
    relative_gap_weight=1.0,
    max_dd_gap_weight=10.0,
    cluster_relative_gap_weight=1.0,
    cluster_participation_gap_weight=100.0,
    cluster_max_dd_gap_weight=5.0,
)

scores = score_state_conditioned_objective(
    window_metrics_by_candidate,
    cluster_metrics_by_candidate=cluster_metrics_by_candidate,
    config=objective_config,
)
summary = summarize_state_conditioned_objective(scores)
review = render_state_conditioned_objective_review(scores, summary)

scores.to_csv(CARD / "objective_scores.csv", index=False)
(CARD / "objective_review.md").write_text(review, encoding="utf-8")

display(scores)
print(json.dumps(summary, indent=2))

,candidate,window_count,window_pass_count,window_gate,cluster_count,cluster_gate,objective_pass,objective_loss,relative_loss,drawdown_loss,participation_loss,min_relative_return,max_max_dd,mean_avg_pos,worst_window,worst_cluster
0,selected_router_no_equity_dd,4,0,False,3,False,False,967.395192,689.739584,267.011898,10.643711,-331.926371,31.932760,0.248807,strict_oos,C003
1,selected_router_dd12_22_cap25,4,0,False,3,False,False,984.291625,710.756855,262.595402,10.939368,-357.211523,31.932760,0.247346,strict_oos,C003
2,selected_router_dd10_20_cap20,4,0,False,3,False,False,1149.417935,891.267058,241.470244,16.680633,-522.594303,31.978926,0.236082,strict_oos,C003
3,selected_router_dd08_18_cap15,4,0,False,3,False,False,1391.042995,1110.092326,245.970897,34.979772,-699.576589,31.086960,0.194262,strict_oos,C003


{
  "decision": "no_objective_pass_candidate",
  "selected_candidate": "selected_router_no_equity_dd",
  "selected_objective_loss": 967.395192410779,
  "selected_window_gate": false,
  "selected_cluster_gate": false,
  "next_mechanism": "train_state_conditioned_multi_objective",
  "required_terms": [
    "walkforward_relative_return_gap_penalty",
    "walkforward_drawdown_gap_penalty",
    "high_vol_rebound_participation_floor",
    "downtrend_drawdown_state_cap"
  ]
}


## 4. Decision

Write the machine-readable selection. The notebook remains the primary single-run conclusion.

In [5]:
selected = scores.iloc[0].to_dict() if not scores.empty else {}
selected_candidate = str(selected.get("candidate", ""))
selected_window_metrics = window_df[window_df["candidate"] == selected_candidate].to_dict("records")
selected_cluster_metrics = cluster_df[cluster_df["candidate"] == selected_candidate].to_dict("records")

if bool(selected.get("objective_pass", False)):
    decision = "objective_pass_promotion_review_required"
    promotion_allowed = False
    blocker = "Objective screen passed, but promotion still requires independent replay/promotion review."
else:
    decision = "diagnostic_no_objective_pass_no_promotion"
    promotion_allowed = False
    blocker = "No candidate jointly cleared +30pp relative return, <=15% MaxDD, and poor-window cluster participation gates."

selection = {
    "experiment": "state_conditioned_participation_drawdown_objective_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_experiments": [
        "deep_cap_failure_attribution_v1",
        "state_conditioned_objective_screen_v1",
        "state_conditioned_rebound_risk_router_v1",
    ],
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "state_conditioned_objective_screen_v1",
    "role": "diagnose",
    "changed_layer": ["model_selection", "reward", "router_scoring"],
    "decision": decision,
    "promotion_allowed": promotion_allowed,
    "full_gate": bool(selected.get("objective_pass", False)),
    "promotion_blocker": blocker,
    "selected_candidate": selected_candidate,
    "selected_row": selected,
    "objective_summary": summary,
    "selected_window_metrics": selected_window_metrics,
    "selected_cluster_metrics": selected_cluster_metrics,
    "external_sources": [ref["url"] for ref in EXTERNAL_REFERENCES],
    "sidecars": [
        "candidate_configs.csv",
        "candidate_path_manifest.csv",
        "window_metrics.csv",
        "cluster_metrics.csv",
        "objective_scores.csv",
        "objective_review.md",
        "summary.json",
    ],
    "acceptance_gate_for_next_run": {
        "relative_return_min_pp": TARGET_EXCESS,
        "max_dd_max_pct": TARGET_MAX_DD,
        "fixed_split_required": True,
        "walkforward_required": True,
        "objective_pass_required_before_ppo": True,
        "avoid_participation_only_or_drawdown_only_objective": True,
    },
}

summary_doc = {
    "experiment": selection["experiment"],
    "decision": decision,
    "selected_candidate": selected_candidate,
    "selected_objective_loss": selected.get("objective_loss"),
    "selected_window_gate": selected.get("window_gate"),
    "selected_cluster_gate": selected.get("cluster_gate"),
    "selected_worst_window": selected.get("worst_window"),
    "selected_worst_cluster": selected.get("worst_cluster"),
    "conclusion": [
        "Adding a path-equity drawdown controller lowers risk for some windows but does not create a full objective-pass candidate.",
        "The selected row should be treated as a training-objective diagnostic, not a runtime profile.",
        "Next PPO work should encode both high-vol rebound participation and downtrend/equity drawdown penalties, then validate by walk-forward checkpoint selection.",
    ],
}

(CARD / "selection.json").write_text(json.dumps(selection, indent=2, ensure_ascii=True), encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(summary_doc, indent=2, ensure_ascii=True), encoding="utf-8")
print(json.dumps(selection, indent=2))

{
  "experiment": "state_conditioned_participation_drawdown_objective_v1",
  "created_at": "2026-05-25T16:25:33.351064+00:00",
  "source_experiments": [
    "deep_cap_failure_attribution_v1",
    "state_conditioned_objective_screen_v1",
    "state_conditioned_rebound_risk_router_v1"
  ],
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "state_conditioned_objective_screen_v1",
  "role": "diagnose",
  "changed_layer": [
    "model_selection",
    "reward",
    "router_scoring"
  ],
  "decision": "diagnostic_no_objective_pass_no_promotion",
  "promotion_allowed": false,
  "full_gate": false,
  "promotion_blocker": "No candidate jointly cleared +30pp relative return, <=15% MaxDD, and poor-window cluster participation gates.",
  "selected_candidate": "selected_router_no_equity_dd",
  "selected_row": {
    "candidate": "selected_router_no_equity_dd",
    "window_count": 4,
    "window_pass_count": 0,
    "window_gate": false,
    "cluster_count

## Conclusion

This notebook should answer one narrow question: whether adding an explicit drawdown controller term to the state-conditioned participation router is enough to clear the hard target before training. If the answer is still no, the next training branch needs a true joint objective rather than another path-level cap or rebound-only participation rule.